# Part 1 -- Diagnosing retrieval failures

A small RAG pipeline, measured on NeoQA, a fully fictional newswire (why
NeoQA: see the README).

Code cells are excerpts from the real pipeline; outputs are pasted from the
actual runs. Question examples are paraphrased, never quoted -- the dataset
ships encrypted to stay out of training crawls, and we keep it that way.

## 1. What NeoQA looks like

Everything in it is invented by an LLM: fictional cities, companies, people.
The corpus is built as timelines -- each timeline follows one story through a
sequence of events, and every event is published as several reworded article
copies: same event, different words. Each question ships with a list of every
article set that counts as complete evidence for it.

In [ ]:
articles  = load_articles("dev")        # id, timeline, event, text
questions = load_answerable("dev")      # question, evidence sets

show_shape(articles, questions)

corpus     360 articles = 3 timelines x 10 events x 12 reworded copies
questions  266 answerable = 156 multi-hop + 110 time-span
           articles needed: 2 (243 questions) / 1 (22) / 3 (1)
           complete evidence sets listed per question: mean 14.2, min 1, max 72

Multi-hop: the answer needs facts from two articles (different events) linked
by a shared entity. Time-span: the answer is a duration computed from dates
across articles.

**The evidence-set list is the detail that matters most later.** For a
two-article question, any pair of copies -- one copy from each of its two
events -- looks like valid evidence. Usually it is not: a copy is a rewording,
and rewordings drop details. Only the pairs where both copies carry every
needed fact answer the question, and the dataset lists exactly those pairs.
Copies look alike; they are not interchangeable.

In [ ]:
q = questions[i]        # one of the 266; paraphrased in the output
show(q)

question (paraphrased): why did a product-fraud tracking effort stay
unreliable in the area where an off-road vehicle's field tests were suspended
over poor terrain performance?

needs two events:            the tracking effort (event 3) + the field tests (event 8)
possible copy-pairs:         12 x 12 = 144
listed as complete evidence: 20 pairs

## 2. What counts as a failure?

The pipeline: embed each article whole, in one pass; rank all articles by
cosine similarity to the question; keep the top-10. A question succeeds if
some complete evidence set has all its articles inside the top-10. It fails
if no listed set completes there.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Alibaba-NLP/gte-modernbert-base")  # 149M parameters
vecs = model.encode(texts, normalize_embeddings=True)           # 360 whole articles
                                                                # (longest: 628 tokens,
                                                                #  window: 8192 -- fits)
def top10(question):
    q = model.encode([question], normalize_embeddings=True)
    return (q @ vecs.T)[0].argsort()[::-1][:10]                 # 10 best articles

show(top10(q))   # the question from above

rank  1  health_12_0-ev8-0    cos 0.749   event 8, copy 0
rank  2  health_12_0-ev8-10   cos 0.749   event 8, copy 10
rank  3  health_12_0-ev8-1    cos 0.743   event 8, copy 1
rank  4  health_12_0-ev8-4    cos 0.734   event 8, copy 4
rank  5  health_12_0-ev4-6    cos 0.728   event 4, copy 6
rank  6  health_12_0-ev4-9    cos 0.725   event 4, copy 9
rank  7  health_12_0-ev3-7    cos 0.713   event 3, copy 7
rank  8  health_12_0-ev8-7    cos 0.706   event 8, copy 7
rank  9  health_12_0-ev8-11   cos 0.698   event 8, copy 11
rank 10  health_12_0-ev3-1    cos 0.692   event 3, copy 1

Both needed events are in the top-10 several times over -- as copies. Whether
the question passes depends on whether some pair in there is one of its
listed sets. Score it the obvious way instead -- is the one official evidence
pair fully in the top-10? -- and the count comes out very different:

In [ ]:
covered  = lambda combo: max(art_rank[a] for a in combo) <= 10   # whole set in top-10
old_fail = sum(not covered(q.official_pair) for q in questions)
new_fail = sum(not any(covered(c) for c in q.valid_sets) for q in questions)

the example question, scored:
  official pair   event 3 copy 4 (rank 16) + event 8 copy 3 (rank 11)   not covered
  a listed set    event 3 copy 7 (rank 7)  + event 8 copy 0 (rank 1)    covered -> pass

score against the official pair only:  216 of 266 fail
score against every listed set:        171 of 266 fail     (45 rescued, 21%)

**Score against every valid evidence set before counting failures.** Much of
our failure count was our own bookkeeping, not retrieval misses.

## 3. How bad are the real failures?

Not every failure is equally far from working. For each failing question,
find its best listed set -- the one whose worst-ranked article sits highest
-- and note where that set would complete:

In [ ]:
def severity(q):
    if not any_needed_article_in_top10(q):  return "found-nothing"
    best = min(worst_rank(c) for c in q.valid_sets)   # the closest set
    return "near-miss" if best <= 20 else "deep"

severity of the 171 failures
  found-nothing   30    no needed article anywhere in the top-10
  deep            85    the closest set completes only past rank 20
  near-miss       56    the closest set completes by rank 11-20 -- a nudge away

(To be continued: why the real ones fail, and which fix is worth building.)